## API Key: BU9SnjBmaohN4fRI

### Username: aarushi12345
### Password: @StockTrades123

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Percentile Smirk Strategy for earnings day (ORCL, PLTR) using iVolatility.
- Underlier: iVol /equities/eod/stock-prices FIRST; Yahoo fallback ONLY if iVol missing.
- IV: iVol raw-iv FIRST, then single-stock-option; ONLY if IV missing -> compute from NBBO mid via Black–Scholes.
- Option selection: ~30 DTE, ATM call (Δ≈+0.50) & OTM put (Δ≈−0.20), closest available.
- Signal: smirk = IV(OTM put) - IV(ATM call)
- Decision by percentile vs previous 10 calendar days of business days:
    <= 15th percentile -> buy call
    >= 85th percentile -> buy put
    else               -> no-trade
"""

import os, time, math, json, datetime as dt
from typing import Optional, Dict, Any, List, Tuple
import pandas as pd

# ---------- Optional Yahoo fallback (UNDERLIER ONLY, LAST RESORT) ----------
try:
    import yfinance as yf
except Exception:
    yf = None

# ---------- iVolatility official client ----------
import ivolatility as ivol

# ---------- Credentials ----------
IV_API_KEY  = os.getenv("IV_API_KEY",  "BU9SnjBmaohN4fRI")
IV_USERNAME = os.getenv("IV_USERNAME", "aarushi12345")
IV_PASSWORD = os.getenv("IV_PASSWORD", "@StockTrades123")

def _login(use_api=True):
    if use_api and IV_API_KEY:
        ivol.setLoginParams(apiKey=IV_API_KEY)
        return "api"
    ivol.setLoginParams(username=IV_USERNAME, password=IV_PASSWORD)
    return "userpass"

LOGIN_MODE = _login(True)

# ---------- Strategy knobs ----------
TARGET_DTE = 30
PRIMARY_DTE_WIN   = (20, 45)
SECONDARY_DTE_WIN = (10, 90)
CALL_TARGET_DELTA = 0.50
PUT_TARGET_DELTA  = -0.20
R = 0.03  # risk-free rate
Q = 0.0   # dividend yield

# ---------- Your jobs / earnings dates ----------
JOBS = [
    ("ORCL", "2025-06-11"),
    ("PLTR", "2025-08-04"),
    ("NFLX", "2025-01-21"),
    ("CRM",  "2024-05-30"),
    ("SHOP", "2024-05-08"),
    ("KO",   "2025-07-22"),
    ("PG",   "2025-07-29"),
]

# ---------- Date helpers ----------
def to_date(s: str) -> dt.date:
    return dt.datetime.strptime(s, "%Y-%m-%d").date()

def clamp_bday(d: dt.date) -> dt.date:
    while d.weekday() >= 5:
        d -= dt.timedelta(days=1)
    return d

def prev_bday(d: dt.date) -> dt.date:
    if d.weekday()==0: return d-dt.timedelta(days=3)
    if d.weekday()==6: return d-dt.timedelta(days=2)
    return d-dt.timedelta(days=1)

def business_days_in_range(end_date: dt.date, days_back: int) -> List[dt.date]:
    dates=[]
    d = end_date - dt.timedelta(days=1)
    stop = end_date - dt.timedelta(days=days_back)
    while d >= stop:
        if d.weekday() < 5:
            dates.append(d)
        d -= dt.timedelta(days=1)
    dates.sort()
    return dates

# ---------- Black–Scholes (for IV backsolve ONLY if IV is missing) ----------
def _phi(x): return 0.5*(1.0+math.erf(x/math.sqrt(2.0)))
def _d1(S,K,r,q,T,v): return (math.log(S/K)+(r-q+0.5*v*v)*T)/(v*math.sqrt(T))
def bs_price(cp,S,K,r,q,T,v):
    if S<=0 or K<=0 or T<=0 or v<=0: return float("nan")
    d1=_d1(S,K,r,q,T,v); d2=d1-v*math.sqrt(T)
    return S*math.exp(-q*T)*_phi(d1)-K*math.exp(-r*T)*_phi(d2) if cp=="C" else K*math.exp(-r*T)*_phi(-d2)-S*math.exp(-q*T)*_phi(-d1)
def bs_delta(cp,S,K,r,q,T,v):
    d1=_d1(S,K,r,q,T,v)
    return math.exp(-q*T)*_phi(d1) if cp=="C" else -math.exp(-q*T)*_phi(-d1)
def solve_iv(cp,S,K,r,q,T,mid):
    if mid is None or mid<=0: return None
    lo,hi=1e-4,5.0
    f=lambda v: bs_price(cp,S,K,r,q,T,v)-mid
    flo,fhi=f(lo),f(hi); tries=0
    while flo*fhi>0 and tries<10: hi*=2; fhi=f(hi); tries+=1
    if flo*fhi>0: return None
    for _ in range(100):
        m=0.5*(lo+hi); fm=f(m)
        if abs(fm)<1e-7 or (hi-lo)<1e-6: return max(m,1e-6)
        if flo*fm<=0: hi,fhi=m,fm
        else: lo,flo=m,fm
    return None

# ---------- Rate limiting + simple cache (reduce 429) ----------
class RateLimiter:
    def __init__(self, min_interval=1.0, max_retries=6, backoff_base=1.6):
        self.min_interval = float(min_interval)
        self.max_retries  = int(max_retries)
        self.backoff_base = float(backoff_base)
        self._tlast = 0.0
    def _slot(self):
        now = time.monotonic()
        delta = now - self._tlast
        if delta < self.min_interval:
            time.sleep(self.min_interval - delta)
        self._tlast = time.monotonic()
    def call(self, path: str, **params):
        getter = ivol.setMethod(path)
        global LOGIN_MODE
        for k in range(self.max_retries):
            try:
                self._slot()
                return getter(**params)
            except Exception as e:
                msg = str(e)
                if ("429" in msg) or ("Too Many Requests" in msg):
                    if k == 0 and LOGIN_MODE == "api":
                        LOGIN_MODE = _login(False)
                        continue
                    time.sleep(self.backoff_base**k)
                    continue
                if k == self.max_retries-1:
                    return None
                time.sleep(0.5)

rl = RateLimiter(min_interval=1.0, max_retries=6, backoff_base=1.6)
_cache: Dict[Tuple[str, tuple], Any] = {}

def cached_call(path: str, **params):
    key = (path, tuple(sorted(params.items())))
    if key in _cache: return _cache[key]
    df = rl.call(path, **params)
    _cache[key] = df
    return df

# ---------- UNDERLIER prices (iVol FIRST; Yahoo ONLY if iVol missing) ----------
def iv_close(symbol: str, date: dt.date) -> Optional[float]:
    df = cached_call("/equities/eod/stock-prices", symbol=symbol, from_=date.isoformat(), to=date.isoformat())
    try:
        if df is not None and len(df)>0:
            for c in ("close","Close","closePrice","settlement","Adjusted close","Unadjusted close"):
                if c in df.columns and pd.notna(df.iloc[-1][c]): return float(df.iloc[-1][c])
    except Exception: pass
    return None

def yh_close(symbol: str, date: dt.date) -> Optional[float]:
    if yf is None: return None
    try:
        start=(date-dt.timedelta(days=10)).isoformat()
        end  =(date+dt.timedelta(days=1)).isoformat()
        hist=yf.Ticker(symbol).history(start=start, end=end, interval="1d", auto_adjust=False)
        if hist.empty: return None
        sub=hist[hist.index.date<=date]
        return float(sub["Close"].iloc[-1]) if not sub.empty else None
    except Exception: return None

def prev_and_trade_close(symbol: str, tdate: dt.date):
    d  = clamp_bday(tdate)
    dp = prev_bday(d)
    # iVol FIRST, Yahoo ONLY if iVol missing
    prev_c  = iv_close(symbol, dp) or yh_close(symbol, dp)
    trade_c = iv_close(symbol, d)  or yh_close(symbol, d)
    return prev_c, trade_c

# ---------- Column + NBBO helpers ----------
def pick_col(df: pd.DataFrame, choices: List[str]) -> Optional[str]:
    for c in choices:
        if c in df.columns: return c
    return None

def get_mid(row: pd.Series) -> Optional[float]:
    for b,a in (("bid","ask"),("Bid","Ask"),("bidPrice","askPrice"),
                ("bestBid","bestAsk"),("nbboBid","nbboAsk"),
                ("bid_1545","ask_1545")):
        if b in row and a in row:
            try:
                bid=float(row[b]); ask=float(row[a])
                if bid>0 and ask>0 and ask>=bid: return 0.5*(bid+ask)
            except: pass
    for c in ("last","Last","close","closePrice","price","Price"):
        if c in row:
            try:
                v=float(row[c]); 
                if v>0: return v
            except: pass
    return None

# ---------- API wrappers (ENDPOINTS UNCHANGED) ----------
def get_series_on_date(symbol: str, date: dt.date, exp_from: Optional[dt.date], exp_to: Optional[dt.date]) -> pd.DataFrame:
    if exp_from and exp_to:
        return cached_call("/equities/eod/option-series-on-date",
                           symbol=symbol, date=date.isoformat(),
                           expFrom=exp_from.isoformat(), expTo=exp_to.isoformat())
    else:
        return cached_call("/equities/eod/option-series-on-date",
                           symbol=symbol, date=date.isoformat())

def get_single_rawiv(option_symbol: str, date: dt.date) -> Optional[pd.DataFrame]:
    return cached_call("/equities/eod/single-stock-option-raw-iv",
                       symbol=option_symbol, from_=date.isoformat(), to=date.isoformat())

def get_single_option(option_symbol: str, date: dt.date) -> Optional[pd.DataFrame]:
    return cached_call("/equities/eod/single-stock-option",
                       symbol=option_symbol, from_=date.isoformat(), to=date.isoformat())

def get_nbbo_chain(symbol: str, date: dt.date) -> Optional[pd.DataFrame]:
    df = cached_call("/equities/eod/options-nbbo", symbol=symbol, date=date.isoformat())
    if df is not None and len(df)>0: return df
    df = cached_call("/equities/eod/options-nbbo", symbol=symbol, from_=date.isoformat(), to=date.isoformat())
    if df is not None and len(df)>0: return df
    df = cached_call("/equities/eod/options-nbbo_1545", symbol=symbol, date=date.isoformat())
    if df is not None and len(df)>0: return df
    return cached_call("/equities/eod/options-nbbo_1545", symbol=symbol, from_=date.isoformat(), to=date.isoformat())

# ---------- Selection + evaluation ----------
def pick_candidates(series_df: pd.DataFrame, symbol: str, trade_date: dt.date, spot: float,
                    dte_window: Tuple[int,int], per_side_limit: int = 4) -> Tuple[List[str], List[str]]:
    if series_df is None or series_df.empty:
        return [], []
    sym_col = pick_col(series_df, ["OptionSymbol","optionSymbol","opt_symbol","symbol","ticker","option symbol"])
    cp_col  = pick_col(series_df, ["callPut","Call/Put","cp","type"])
    k_col   = pick_col(series_df, ["strike","strikePrice","strike price"])
    exp_col = pick_col(series_df, ["expirationDate","expiration","expDate"])
    if not sym_col or not cp_col or not k_col:
        return [], []

    df = series_df.copy()
    if exp_col:
        df["_exp"] = pd.to_datetime(df[exp_col]).dt.date
    else:
        def parse_occ(s):
            try:
                p=str(s).split()[1][:6]; return dt.datetime.strptime(p,"%y%m%d").date()
            except: return None
        df["_exp"] = df[sym_col].astype(str).apply(parse_occ)

    df["_dte"] = df["_exp"].apply(lambda e: (e - trade_date).days if pd.notna(e) else None)
    df = df[(df["_dte"]>=dte_window[0]) & (df["_dte"]<=dte_window[1])]
    if df.empty: return [], []

    df["_distATM"] = (df[k_col].astype(float) - float(spot)).abs()
    calls = df[df[cp_col].astype(str).str.upper().str.startswith("C")].sort_values(["_dte","_distATM"]).head(per_side_limit)
    puts  = df[df[cp_col].astype(str).str.upper().str.startswith("P")].sort_values(["_dte","_distATM"]).head(per_side_limit)

    return (calls[sym_col].astype(str).unique().tolist(),
            puts[sym_col].astype(str).unique().tolist())

def evaluate_list(option_symbols: List[str], cp: str, trade_date: dt.date, S: float) -> Optional[Dict[str,Any]]:
    """
    IV retrieval order (as you requested):
      1) /single-stock-option-raw-iv
      2) /single-stock-option
      3) IF IV still missing -> compute from NBBO mid (Black–Scholes) using (2)'s row
    """
    best=None; best_score=1e9
    for sym in option_symbols:
        iv_val=None; delta=None; exp=None; K=None

        # 1) Raw IV (preferred)
        df_r = get_single_rawiv(sym, trade_date)
        if df_r is not None and len(df_r)>0:
            d = df_r.iloc[-1].to_dict()
            iv_val = next((float(d[k]) for k in ("iv","ivMid","impliedVol","IV") if k in d and d[k] is not None), None)
            delta  = next((float(d[k]) for k in ("delta","Delta","optionDelta") if k in d and d[k] is not None), None)
            exp    = next((str(d[k])   for k in ("expiration","expDate") if k in d and d[k] is not None), None)
            K      = next((float(d[k]) for k in ("strike","strikePrice") if k in d and d[k] is not None), None)

        # 2) Single-stock (if anything missing)
        if iv_val is None or delta is None or exp is None or K is None:
            df_s = get_single_option(sym, trade_date)
            if df_s is not None and len(df_s)>0:
                d = df_s.iloc[-1].to_dict()
                iv_val = iv_val or next((float(d[k]) for k in ("iv","ivMid","impliedVol","IV") if k in d and d[k] is not None), None)
                delta  = delta  or next((float(d[k]) for k in ("delta","Delta","optionDelta") if k in d and d[k] is not None), None)
                exp    = exp    or next((str(d[k])   for k in ("expiration","expDate") if k in d and d[k] is not None), None)
                K      = K      or next((float(d[k]) for k in ("strike","strikePrice") if k in d and d[k] is not None), None)

        # 3) Backsolve IV ONLY if still missing (using NBBO mid from the single-stock row)
        if (iv_val is None or delta is None) and (exp is not None and K is not None):
            df_one = get_single_option(sym, trade_date)
            if df_one is not None and len(df_one)>0:
                r = df_one.iloc[-1]
                mid = get_mid(r)
                if mid:
                    exp_d = dt.datetime.strptime(str(exp)[:10], "%Y-%m-%d").date()
                    dte = max(1, (exp_d - trade_date).days)
                    T = dte/365.0
                    ivv = solve_iv(cp, S, float(K), R, Q, T, mid)
                    if ivv:
                        iv_val = iv_val or ivv
                        delta  = delta  or bs_delta(cp, S, float(K), R, Q, T, ivv)

        if iv_val is None or delta is None or exp is None:
            continue

        dte = (dt.datetime.strptime(str(exp)[:10], "%Y-%m-%d").date() - trade_date).days
        target = CALL_TARGET_DELTA if cp=="C" else PUT_TARGET_DELTA
        score = abs(delta - target) + 0.01*abs(dte - TARGET_DTE)
        if score < best_score:
            best_score = score
            best = {"ticker": sym,
                    "iv": float(iv_val)*100.0 if iv_val<2.0 else float(iv_val),
                    "delta": float(delta),
                    "dte": int(dte)}
    return best

# ---------- Smirk for a single date ----------
def smirk_for_date(symbol: str, trade_date: dt.date) -> Dict[str, Any]:
    prev_c, trade_c = prev_and_trade_close(symbol, trade_date)
    S = trade_c or prev_c
    if not S:
        return {"smirk": None, "atm_call": None, "otm_put": None}

    exp_from = trade_date + dt.timedelta(days=TARGET_DTE-15)
    exp_to   = trade_date + dt.timedelta(days=TARGET_DTE+15)
    series   = get_series_on_date(symbol, trade_date, exp_from, exp_to)
    if series is None or series.empty:
        series = get_series_on_date(symbol, trade_date, None, None)

    call_syms, put_syms = pick_candidates(series, symbol, trade_date, S, PRIMARY_DTE_WIN, per_side_limit=4)
    if not call_syms and not put_syms:
        call_syms, put_syms = pick_candidates(series, symbol, trade_date, S, SECONDARY_DTE_WIN, per_side_limit=4)

    atm_call = evaluate_list(call_syms, "C", trade_date, S) if call_syms else None
    otm_put  = evaluate_list(put_syms,  "P", trade_date, S) if put_syms  else None

    iv_call = atm_call["iv"] if atm_call else None
    iv_put  = otm_put["iv"]  if otm_put  else None
    smk = (iv_put - iv_call) if (iv_call is not None and iv_put is not None) else None
    return {"smirk": smk, "atm_call": atm_call, "otm_put": otm_put}

# ---------- Percentile logic ----------
def percentile_rank(x: float, arr: List[float]) -> Optional[float]:
    if x is None or not arr:
        return None
    xs = sorted(v for v in arr if v is not None)
    if not xs:
        return None
    count = sum(1 for v in xs if v <= x)
    return 100.0 * count / len(xs)

def analyze_percentile(symbol: str, earnings_date: str, lookback_days: int = 10,
                       bottom_pct: float = 15.0, top_pct: float = 85.0) -> Dict[str, Any]:
    tdate = clamp_bday(to_date(earnings_date))
    prev_c, trade_c = prev_and_trade_close(symbol, tdate)
    move_pct = (100.0*(trade_c - prev_c)/prev_c) if (prev_c and trade_c) else None

    today = smirk_for_date(symbol, tdate)
    smk_today = today["smirk"]

    lb_days = business_days_in_range(tdate, lookback_days)
    lb_smirks=[]
    for d in lb_days:
        val = smirk_for_date(symbol, d)["smirk"]
        if val is not None:
            lb_smirks.append(val)

    pct = percentile_rank(smk_today, lb_smirks)
    suggestion = "no-trade"
    if pct is not None:
        if pct <= bottom_pct: suggestion = "buy call"
        elif pct >= top_pct:  suggestion = "buy put"

    aligned = None
    if suggestion == "buy call" and move_pct is not None:
        aligned = (move_pct > 0)
    elif suggestion == "buy put" and move_pct is not None:
        aligned = (move_pct < 0)

    return {
        "symbol": symbol,
        "date": tdate.isoformat(),
        "prev_close": prev_c,
        "trade_close": trade_c,
        "earnings_day_move_pct": move_pct,
        "atm_call": today["atm_call"],
        "otm_put": today["otm_put"],
        "smirk_today": smk_today,
        "lookback_days": len(lb_days),
        "lookback_smirks_observed": len(lb_smirks),
        "smirk_percentile": pct,
        "rule": {"bottom_pct": bottom_pct, "top_pct": top_pct, "lookback_calendar_days": lookback_days},
        "suggestion": suggestion,
        "aligned_with_move": aligned
    }

# ---------- Batch run ----------
def run() -> Dict[str, Any]:
    results = []
    for sym, d in JOBS:
        results.append(analyze_percentile(sym, d, lookback_days=30, bottom_pct=10.0, top_pct=90.0))
    return {"results": results}

if __name__ == "__main__":
    print(json.dumps(run(), indent=2))

| **Ticker** | **Date (MM/DD)** | **Direction** |
| ---------- | ---------------- | ------------- |
| **DDOG**   | 10/31            | +23.08%       |
| **NET**    | 11/07            | +16.04%       |
| **AMD**    | 10/29            | +7.42%        |
| **QCOM**   | 11/06            | +6.34%        |
| **MU**     | 11/27            | +5.77%        |
| **AOS**    | 05/28            | –6.30%        |
| **PYPL**   | 11/01            | –6.89%        |
| **EL**     | 11/01            | –18.91%       |
| **CMCSA**  | 10/30            | –6.12%        |
| **SWKS**   | 11/06            | –7.53%        |
| **AAPL**   | 11/02            | +1.23%        |
| **KO**     | 10/24            | +2.32%        |
| **PEP**    | 10/10            | +1.98%        |
| **CMG**    | 10/24            | +3.09%        |
| **VZ**     | 10/24            | +3.52%        |
| **GOOGL**  | 10/26            | –0.96%        |
| **META**   | 10/25            | –1.02%        |
| **UNH**    | 10/13            | –1.25%        |
| **JNJ**    | 10/17            | –0.56%        |
| **NFLX**   | 10/18            | –0.20%        |
